# Track A — S24 SD LoRA VS Generation (Colab)

**전제조건**: 런타임 유형 → GPU (T4 이상)

**순서**: [1] → [2] → [3] → [4] → [5] → [6] → [7]

In [ ]:
# [1] GPU 확인
import torch
assert torch.cuda.is_available(), 'GPU 없음: 런타임 > 런타임 유형 변경 > T4 GPU'
print(f'torch  {torch.__version__}')
print(f'GPU    {torch.cuda.get_device_name(0)}')

In [ ]:
# [2] 환경 세팅
# Colab 환경: torch 2.11.0+cu130 / torchvision 0.26.0+cu128 / torchaudio cu128
# torchvision/torchaudio 모두 cu128 빌드 → cu130 torch와 CUDA major 불일치
import subprocess, os, re, importlib.util

# ── 1. torchvision CUDA 버전 체크 비활성화 ───────────────────────────────────
_ext = '/usr/local/lib/python3.12/dist-packages/torchvision/extension.py'
with open(_ext) as f: _s = f.read()
_s = _s.replace('def # _check_cuda_version()  # patched:', 'def _check_cuda_version():')
_s = re.sub(r'if t_major != tv_major:', 'if False:  # patched', _s)
_s = re.sub(r'^_check_cuda_version\(\)\s*$', '# _check_cuda_version()  # patched', _s, flags=re.MULTILINE)
with open(_ext, 'w') as f: f.write(_s)
print('torchvision patched')

# ── 2. torchaudio CUDA 버전 체크 비활성화 ────────────────────────────────────
_ta = '/usr/local/lib/python3.12/dist-packages/torchaudio/_extension/utils.py'
if os.path.exists(_ta):
    with open(_ta) as f: _s = f.read()
    _s = re.sub(r'if ta_version != t_version:', 'if False:  # patched', _s)
    with open(_ta, 'w') as f: f.write(_s)
    print('torchaudio patched')

# ── 3. peft / diffusers / accelerate 설치 (--no-deps: torch 버전 유지) ───────
subprocess.run(['pip', 'install', '-q', '--no-deps',
                'peft', 'diffusers', 'accelerate'], check=True)

# ── 4. peft 패치 — torchao 구버전(0.10.0)에서 raise 대신 return False ────────
_spec = importlib.util.find_spec('peft')
if _spec:
    _pu = os.path.join(os.path.dirname(_spec.origin), 'import_utils.py')
    with open(_pu) as f: _s = f.read()
    _p = _s.replace(
        'if torchao_version < TORCHAO_MINIMUM_VERSION:',
        'if torchao_version < TORCHAO_MINIMUM_VERSION:\n        return False  # patched'
    )
    if _p != _s:
        with open(_pu, 'w') as f: f.write(_p)
        print('peft patched OK')
    else:
        print('peft: already patched or pattern not found')

# ── 5. transformers + 의존성 설치 ─────────────────────────────────────────────
subprocess.run(['pip', 'install', '-q', '-U',
                'transformers', 'tokenizers', 'huggingface-hub', 'safetensors',
                'packaging', 'h5py', 'regex', 'tqdm', 'filelock', 'requests'], check=True)

# ── 6. 검증 (fresh subprocess) ───────────────────────────────────────────────
r = subprocess.run(['python', '-c', '''
import torch, torchvision, torchaudio
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig, get_peft_model
import transformers
print(f"torch        {torch.__version__}")
print(f"torchvision  {torchvision.__version__}")
print(f"torchaudio   {torchaudio.__version__}")
print(f"diffusers    OK")
print(f"peft         OK")
print(f"transformers {transformers.__version__}")
print("ALL OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('=== ERROR ===')
    print(r.stderr[-3000:])
    raise RuntimeError('환경 검증 실패 — 위 에러 확인 필요')

In [ ]:
# [3] 코드 clone / pull (S24 .npz 포함)
import os, subprocess

REPO    = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

def _clone():
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

if os.path.exists(WORKDIR):
    if not os.path.exists(os.path.join(WORKDIR, '.git')):
        subprocess.run(['rm', '-rf', WORKDIR], check=True)
        _clone()
    else:
        r = subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', 'main'])
        if r.returncode != 0:
            subprocess.run(['rm', '-rf', WORKDIR], check=True)
            _clone()
else:
    _clone()

os.chdir(WORKDIR)
subprocess.run(['git', 'log', '--oneline', '-3'])

npz = [f for f in os.listdir('preproc_vs_re') if 'subj_24' in f and f.endswith('.npz')]
ckpt = 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon/subj24_best.pt'
print(f'S24 .npz: {len(npz)}개,  SupCon ckpt: {os.path.exists(ckpt)}')

In [ ]:
# [4] Google Drive 마운트 (체크포인트 저장)
from google.colab import drive
drive.mount('/content/drive')

CKPT_DRIVE = '/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen'
os.makedirs(CKPT_DRIVE, exist_ok=True)

dst = '/content/vsvi_project/checkpoints_vsre_lora_gen'
if not os.path.exists(dst):
    os.symlink(CKPT_DRIVE, dst)
print(f'checkpoints_vsre_lora_gen -> {CKPT_DRIVE}')

In [ ]:
# [5] 최종 확인
import os, torch
W = '/content/vsvi_project'
checks = {
    'CUDA':        torch.cuda.is_available(),
    'npz 10개':    len([f for f in os.listdir(f'{W}/preproc_vs_re') if 'subj_24' in f and f.endswith('.npz')]) == 10,
    'SupCon ckpt': os.path.exists(f'{W}/checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon/subj24_best.pt'),
    'images':      os.path.exists(f'{W}/preproc_data_vi/images/01.png'),
    'ckpt_root':   os.path.islink(f'{W}/checkpoints_vsre_lora_gen') or os.path.isdir(f'{W}/checkpoints_vsre_lora_gen'),
}
for k, v in checks.items():
    print(f'  [{"OK" if v else "FAIL"}] {k}')
assert all(checks.values()), 'FAIL 항목 해결 후 재실행'

In [ ]:
# [6] S24 LoRA r=16 학습 (T4 기준 약 8~12시간)
# --fp16 --grad_ckpt: T4 14.5GB 메모리 절약
import os
os.chdir('/content/vsvi_project')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
!python -u train_vs_re_lora_gen.py \
    --subject_ids 24 \
    --lora_r 16 \
    --n_eeg_tokens 16 \
    --epochs 100 \
    --batch_size 2 \
    --fp16 \
    --grad_ckpt \
    --img_root preproc_data_vi/images \
    --supcon_ckpt checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon \
    --ckpt_root checkpoints_vsre_lora_gen

In [ ]:
# [7] S24 LoRA r=32 학습 (r=16 완료 후 실행)
!python -u train_vs_re_lora_gen.py \
    --subject_ids 24 \
    --lora_r 32 \
    --n_eeg_tokens 16 \
    --epochs 100 \
    --batch_size 2 \
    --fp16 \
    --grad_ckpt \
    --img_root preproc_data_vi/images \
    --supcon_ckpt checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon \
    --ckpt_root checkpoints_vsre_lora_gen

In [ ]:
# [8] 결과 확인
import glob
csvs = glob.glob('/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/**/results_lora_gen.csv', recursive=True)
for p in sorted(csvs):
    print(p)
    with open(p) as f: print(f.read())

In [ ]:
# [9] 결과 → PROGRESS.md 업데이트 + GitHub push
# 사전 준비: Colab 왼쪽 🔑 아이콘 → Secrets → GITHUB_TOKEN 추가 (GitHub PAT)
import os, csv, datetime, subprocess

WORKDIR = '/content/vsvi_project'

# ── 1. 결과 CSV 수집 (로컬 symlink / Drive 직접경로 둘 다 탐색) ───────────────
search_roots = [
    f'{WORKDIR}/checkpoints_vsre_lora_gen',
    '/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen',
]

csvs = []
for root in search_roots:
    if not os.path.exists(root):
        continue
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        for fn in filenames:
            if fn == 'results_lora_gen.csv':
                p = os.path.join(dirpath, fn)
                if p not in csvs:
                    csvs.append(p)
csvs.sort()

print(f'찾은 CSV {len(csvs)}개:')
for p in csvs: print(' ', p)

if not csvs:
    print('\n결과 CSV 없음 — 학습 완료 후 실행하세요')
    raise SystemExit

# ── 2. CSV 파싱 ───────────────────────────────────────────────────────────────
results = {}   # {(sid, lora_r): row_dict}
for p in csvs:
    dirname = os.path.basename(os.path.dirname(p))
    r_val = None
    for part in dirname.split('_'):
        if part.startswith('r') and part[1:].isdigit():
            r_val = int(part[1:])
    with open(p) as f:
        for row in csv.DictReader(f):
            sid = int(row['sid'])
            key = (sid, r_val)
            if key not in results:
                results[key] = {**row, 'lora_r': r_val, 'dirname': dirname}

print('\n파싱 결과:')
for (sid, r), row in sorted(results.items()):
    print(f"  S{sid:02d} r={r}: DINO@1={float(row['top1']):.4f}")

# ── 3. PROGRESS.md 업데이트 텍스트 생성 ──────────────────────────────────────
today = datetime.datetime.now().strftime('%Y-%m-%d')
lines = [
    f'\n## S24 SD LoRA VS Generation 재실행 결과 ({today})\n',
    '\n**SupCon**: `checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon/subj24_best.pt`\n',
    '\n| r | best_ep | DINO@1 | DINO@3 | DINO@5 | dominant | entropy | dirname |\n',
    '|---|---------|--------|--------|--------|----------|---------|---------|',
]
for (sid, r), row in sorted(results.items()):
    lines.append(
        f"\n| {r} | {row['best_ep']} | {float(row['top1']):.4f} | "
        f"{float(row['top3']):.4f} | {float(row['top5']):.4f} | "
        f"{float(row['dominant'])*100:.1f}% | {float(row['entropy']):.3f} | `{row['dirname']}` |"
    )

r16 = results.get((24, 16))
r32 = results.get((24, 32))
top1_16 = f"{float(r16['top1']):.4f}" if r16 else 'TBD'
top1_32 = f"{float(r32['top1']):.4f}" if r32 else 'TBD'
lines += [
    '\n\n**S24 vs S01/S18 비교 (업데이트):**\n',
    '\n| Subject | r=16 DINO@1 | r=32 DINO@1 | Sessions | Note |\n',
    '|---------|-------------|-------------|----------|------|',
    '\n| S01 | 0.3333 | **0.3571** | 9 | S01 best = r=32 |',
    '\n| S18 | **0.2963** | 0.2870 | 8 | S18 best = r=16 |',
    f'\n| S24 | {top1_16} | {top1_32} | 10 | 재실행 (올바른 SupCon) |\n',
    '\n---\n',
]
update_text = ''.join(lines)
print('\n=== PROGRESS.md 추가 내용 ===')
print(update_text)

# ── 4. PROGRESS.md 삽입 ───────────────────────────────────────────────────────
progress_path = f'{WORKDIR}/PROGRESS.md'
with open(progress_path) as f:
    original = f.read()

insert_marker = '\n## Current Status Summary'
insert_pos = original.find(insert_marker)
updated = (original[:insert_pos] + update_text + original[insert_pos:]) if insert_pos != -1 else (original + update_text)
with open(progress_path, 'w') as f:
    f.write(updated)
print('PROGRESS.md 업데이트 완료')

# ── 5. GitHub push ────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    import getpass
    token = getpass.getpass('GitHub PAT 입력: ')

REPO = f'https://{token}@github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
subprocess.run(['git', '-C', WORKDIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', WORKDIR, 'merge', 'origin/main', '--no-edit'], check=True)
subprocess.run(['git', '-C', WORKDIR, 'remote', 'set-url', 'origin', REPO], check=True)
subprocess.run(['git', '-C', WORKDIR, 'config', 'user.email', 'colab@vsvi'], check=True)
subprocess.run(['git', '-C', WORKDIR, 'config', 'user.name', 'Colab'], check=True)
subprocess.run(['git', '-C', WORKDIR, 'add', 'PROGRESS.md'], check=True)
subprocess.run(['git', '-C', WORKDIR, 'commit', '-m', f'S24 LoRA results ({today})'], check=True)
r = subprocess.run(['git', '-C', WORKDIR, 'push', 'origin', 'HEAD:main'], capture_output=True, text=True)
if r.returncode == 0:
    print('GitHub push 완료')
else:
    print('push 실패:', r.stderr[-500:])